# Jev_CeNN — Global Layer 15: Controlled Branch Ablation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Jev_CeNN_Layer15_Branch_Ablation_Colab.ipynb)

## Objective

**How much does each component of trained Jev_CeNN layer 15 contribute to matching the original teacher, task decisions, and inference speed?**

The Colab replaces **one and only one** original **global attention layer, index 15**. The other 27 original encoder attention modules are left unchanged. The teacher and task head remain frozen.

### Comparisons: same trained layer, surgical interventions

| Test | Intervention |
|---|---|
| **native_teacher** | Original ModernBERT attention in all 28 layers |
| **full** | Trained Jev_CeNN FastWindowMemory at index 15, other layers native |
| **manual_full** | Independent reconstruction of full layer forward (must numerically match `full`) |
| **minus_local** | Remove full-context local SDPA and renormalize branch routing |
| **minus_landmarks** | Remove pooled landmark SDPA and renormalize routing |
| **minus_linear** | Remove learned global linear memory and renormalize routing |
| **minus_value** | Remove direct value-feature bypass and renormalize routing |
| **minus_cenn** | Remove additive 5-tap cellular convolution residual |
| **static_routing** | Remove query-conditioned branch selection; keep learned head-wise weights |
| **only_local / only_landmarks / only_linear / only_value** | Single-branch diagnostic **without** CeNN residual |
| **only_cenn** | Standalone learned CeNN convolution on pretrained value features |

**Methodological caveat:** A drop-one intervention is a *conditional marginal effect*. Softmax weights are renormalized among surviving branches. Branch effects are not additive and this is **not a Shapley-value attribution**. Single-branch controls are not independently trained from scratch.

### Correct evaluation

- Train **only layer 15** with frozen pretrained QKV and RoPE against original teacher layer-15 outputs, or load a **matching trained layer-15 checkpoint** from V3.1/Jev_CeNN.
- Freeze every other layer throughout. All ablations start with **identical trained weights**.
- Use disjoint **TRAIN**, DEV-PROBE, DEV-AUDIT, and downstream DEV decision sets. The **official test split is not touched**.
- Validate `manual_full` against the trained `full` forward first. If that parity fails, stop: the ablation results would be untrustworthy.
- Evaluate **token output cosine / NMSE**, **full-model decision agreement / gold accuracy / JS divergence**, route usage, **layer-only GPU latency** and **end-to-end GPU latency**.
- Optional recovery training, separately initialized from the same trained base and selected on **DEV-PROBE only**, answers whether removed branches are *essential* or merely *replaceable after retraining*.
- Save `csv`, `json` and comparison figures. **No Hugging Face publication**: this is an exploratory diagnostic, not a new model.

At W1024 with max length 512, the `local` path is effectively **dense full attention**. Removing it may save time but may also lose content-addressable retrieval.

A good research conclusion is not simply “largest cosine”: it is the **Pareto tradeoff between held-out fidelity and actual GPU speed**, plus downstream task decisions.


In [ ]:
#@title 1. Install pinned Jev_CeNN attention ablation dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Original teacher, one-layer index 15, and ablation settings
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite, decision_distill_loss, guarded_step
from tinycenn_lm.sliding_window_fusion import SlidingWindowFusion

SEED = 2028
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15
FULL_WINDOW = 1024 #@param {type:"integer"}
LANDMARK_STRIDE = 16
FEATURE_DIM = 96
MAX_LANDMARKS = 32
TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 180 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 2 #@param {type:"integer"}
GLOBAL_STEPS = 400 #@param {type:"integer"}
STEPS = GLOBAL_STEPS
CHECK_EVERY = 50
TRAIN_CACHE_BATCHES = 32 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 8 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 8 #@param {type:"integer"}
QUALITY_DECISIONS = 160 #@param {type:"integer"}
LATENCY_REPEATS = 15 #@param {type:"integer"}
LOAD_LAYER15_CHECKPOINT = True #@param {type:"boolean"}
LAYER15_CHECKPOINT = "/content/Jev_CeNN/global_W1024/layer15_W1024.pt" #@param {type:"string"}
RECOVERY_STEPS = 0 #@param {type:"integer"}
RECOVERY_MODES = ("minus_local", "minus_landmarks", "minus_linear",
                  "minus_value", "minus_cenn", "static_routing")
TARGET_COSINE = 0.99
TARGET_NMSE = 0.025
OUTPUT_DIR = Path("/content/Jev_CeNN_Layer15_Ablation")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert GLOBAL_STEPS>0 and BATCH>=1 and LATENCY_REPEATS>=3
assert RECOVERY_STEPS>=0 and QUALITY_DECISIONS>=16
assert 15==LAYER and FULL_WINDOW>=MAX_LEN

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
from tinycenn_lm.standalone_decision import full_attention_indices
GLOBAL_LAYERS = full_attention_indices(teacher)
SLIDING_LAYERS = [i for i in range(len(teacher.encoder.layers)) if i not in GLOBAL_LAYERS]
assert GLOBAL_LAYERS and 15 in GLOBAL_LAYERS and len(GLOBAL_LAYERS)+len(SLIDING_LAYERS)==len(teacher.encoder.layers)
for i in GLOBAL_LAYERS:
    assert tuple(teacher.encoder.layers[i].attn.local_attention) == (-1, -1)
print("Global layers:", GLOBAL_LAYERS, "| native sliding:", SLIDING_LAYERS)
print("Global W:", FULL_WINDOW, "| MAX_LEN:", MAX_LEN)
assert tuple(teacher.encoder.layers[LAYER].attn.local_attention)==(-1,-1)
print("W>=2*MAX_LEN-1 gives dense full-context local attention:", FULL_WINDOW >= 2*MAX_LEN-1)
print("Teacher:", SOURCE_MODEL, "| STUDENT REPLACES ONLY L15 | test remains untouched")


In [ ]:
#@title 3. Case-disjoint TRAIN, PROBE, AUDIT and downstream DEV I/O
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
        # ModernBERT SDPA requires position_ids and sliding_window_mask.
        # Replay the exact teacher call, including all positional/mask kwargs.
        got["call_kwargs"] = {
            key: (value.detach() if torch.is_tensor(value) else value)
            for key, value in kwargs.items()
        }
        if "position_ids" not in got["call_kwargs"] or "sliding_window_mask" not in got["call_kwargs"]:
            raise RuntimeError("Teacher attention hook did not capture ModernBERT SDPA kwargs")
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred = repl(entry["x"], **entry["call_kwargs"])[0]
            require_finite(pred, "all-attention local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/dev decisions:", len(train_items), len(dev_items))
assert len(dev_items) >= BATCH * (PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES), "Increase DEV_CASES for disjoint probe and audit caches"
print("official test remains untouched; probe/audit I/O non-overlapping")


In [ ]:
#@title 4. Exact V3.1 global FastWindowMemory layer (identical to trained model)
class PDeltaV28FastWindowMemory(PDelta3GDN2CLVRAttention):
    """PDelta3-inspired replacement without sequential GDN scan.

    Complexity: local attention O(T*W*d), landmark cross attention
    O(T*ceil(T/stride)*d), and kernel memory O(T*F*d).
    The landmark branch becomes quadratic if stride is fixed while T -> infinity.
    For provably bounded linear scaling, cap its count with adaptive pooling.
    """
    architecture = "pdelta_v28_fast_window_memory"

    def __init__(self, original, *, local_window=96, landmark_stride=16,
                 feature_dim=96, max_landmarks=32):
        super().__init__(
            original, feature_dim=feature_dim, conv_kernel=4,
            chunk_size=32, local_kernel=5,
            local_window=local_window, local_gate_init=0.90,
        )
        self.landmark_stride = int(landmark_stride)
        self.max_landmarks = int(max_landmarks)
        assert self.landmark_stride > 0 and self.max_landmarks > 0
        # Remove obsolete recurrent, causal-convolution and four-way fusion parameters.
        # The inherited methods _local_attention and _global_linear remain in use.
        for name in (
            "wq", "wk", "decay_w", "dt_bias", "erase_w", "erase_b",
            "write_w", "write_b", "route_proj", "route_gate_w",
            "route_gate_b", "q_conv", "k_conv", "v_conv",
            "mix_logits", "mix_gate_w", "local_gate_w",
            "local_gate_b", "log_gain",
        ):
            delattr(self, name)
        # 4.57.6 passes position_ids to ModernBERT, not position_embeddings.
        # Copy the exact pretrained per-layer RoPE instead of silently skipping it.
        self.rotary_emb = copy.deepcopy(original.rotary_emb)
        self.rotary_emb.requires_grad_(False)
        self.layer_idx = int(getattr(original, "layer_id", -1))
        h, d = self.num_heads, self.head_dim
        # [local, compressed landmarks, global linear, direct value].
        # Layer 15 is global: do not initialize with ~90% exclusively local routing.
        self.route_logits = nn.Parameter(
            torch.tensor([1.0, 0.2, 0.2, -2.2]).repeat(h, 1)
        )
        self.route_query = nn.Parameter(torch.zeros(h, 4, d))
        self.local_log_temperature = nn.Parameter(torch.zeros(h))
        self.landmark_log_temperature = nn.Parameter(torch.zeros(h))
        self.conv_gain = nn.Parameter(torch.full((h,), -4.0))
        # Pretrained QKV and Wo are copied from the teacher by BaseLayaReplacementAttention.
        # Freeze QKV and allow a very small Wo calibration.
        for p in self.Wqkv.parameters():
            p.requires_grad_(False)
        for p in self.Wo.parameters():
            p.requires_grad_(True)

    def _local_attention(self, q, k, v, valid):
        """Centered local window; avoid huge empty W×(2W-1) halos when W >= T."""
        b, h, t, d = q.shape
        w = self.local_window
        if w < t:
            return super()._local_attention(q, k, v, valid)
        allowed = valid[:, None, None, :]
        if w < 2*t-1:
            qi = torch.arange(t, device=q.device)[:, None]
            ki = torch.arange(t, device=q.device)[None, :]
            band = ((ki >= qi-(w-1)//2) & (ki <= qi+w//2))
            allowed = allowed & band[None, None]
        z = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=0.0)
        return z * valid[:, None, :, None].to(z.dtype)

    def _landmarks(self, q, k, v, valid):
        b, h, t, d = k.shape
        # Choose at most max_landmarks pooled key/value blocks. This ensures
        # O(T*M) with M <= max_landmarks at all sequence lengths.
        stride = max(self.landmark_stride, math.ceil(t / self.max_landmarks))
        blocks = math.ceil(t / stride)
        pad = blocks * stride - t
        mask = valid[:, None, :, None].to(k.dtype)
        packed_valid = F.pad(valid, (0, pad), value=False).reshape(b, blocks, stride)
        counts = packed_valid.sum(-1)
        divisor = counts.clamp_min(1).to(k.dtype)[:, None, :, None]
        kp = F.pad(k * mask, (0, 0, 0, pad)).reshape(b, h, blocks, stride, d)
        vp = F.pad(v * mask, (0, 0, 0, pad)).reshape(b, h, blocks, stride, d)
        pooled_k = kp.sum(3) / divisor
        pooled_v = vp.sum(3) / divisor
        allowed = (counts > 0)[:, None, None, :]
        temperature = self.landmark_log_temperature.clamp(-0.4, 0.4).exp()
        q_scaled = q * temperature[None, :, None, None]
        # Only T x M attention scores; never a dense T x T matrix.
        return F.scaled_dot_product_attention(
            q_scaled, pooled_k, pooled_v, attn_mask=allowed, dropout_p=0.0
        ) * mask

    def forward(self, hidden_states, position_embeddings=None,
                attention_mask=None, position_ids=None, sliding_window_mask=None,
                **kwargs):
        # Exact teacher QKV + RoPE path; apply rotary embeddings BEFORE attention.
        b, t, _ = hidden_states.shape
        qkv = self.Wqkv(hidden_states).reshape(b, t, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.transpose(3, 1).unbind(dim=2)
        if position_embeddings is None:
            cos, sin = self.rotary_emb(qkv, position_ids=position_ids)
        else:
            cos, sin = position_embeddings
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        valid = _valid_tokens(attention_mask, hidden_states)
        mask = valid[:, None, :, None].to(v.dtype)
        q, k, v = q * mask, k * mask, v * mask
        temperature = self.local_log_temperature.clamp(-0.4, 0.4).exp()
        qlocal = q * temperature[None, :, None, None]
        local = self._local_attention(qlocal, k, v, valid)
        landmarks = self._landmarks(q, k, v, valid)
        global_linear = self._global_linear(q, k, v, mask) * mask
        qnorm = F.normalize(q, dim=-1)
        dynamic = torch.einsum("bhtd,hkd->bhtk", qnorm, self.route_query)
        mixture = torch.softmax(
            self.route_logits.to(q.dtype)[None, :, None, :] + dynamic,
            dim=-1
        )
        fused = (
            mixture[..., 0:1] * local
            + mixture[..., 1:2] * landmarks
            + mixture[..., 2:3] * global_linear
            + mixture[..., 3:4] * v
        )
        local_conv = _depthwise_sequence_conv(v, self.local_weight, causal=False)
        conv_weight = torch.sigmoid(self.conv_gain).to(v.dtype)[None, :, None, None]
        fused = (fused + conv_weight * (local_conv - v)) * mask
        return self.finish(fused, hidden_states)

    def config_dict(self):
        return dict(
            architecture=self.architecture, layer_idx=self.layer_idx,
            local_window=self.local_window, landmark_stride=self.landmark_stride,
            max_landmarks=self.max_landmarks, feature_dim=self.feature_dim,
            global_linear_memory=True, local_windowed_attention=True,
            dense_global_full_attention=self.local_window >= 2 * MAX_LEN - 1, recurrent_scan=False,
            rope_applied=True, upstream_sdpa_kwargs=True,
        )

def build_v28(window):
    return PDeltaV28FastWindowMemory(
        teacher.encoder.layers[LAYER].attn,
        local_window=int(window),
        landmark_stride=LANDMARK_STRIDE,
        max_landmarks=32,
        feature_dim=FEATURE_DIM,
    ).to(device)

def state_cpu(module):
    return {k: p.detach().cpu().clone() for k, p in module.state_dict().items()}

print("V2.8 architecture ready (no GDN scan; bounded local + 32 capped landmarks)")


In [ ]:
#@title 5. Same layer-15 local distillation as original V3.1
def score_local(m):
    return m["nmse"] + 1.50 * (1.0 - m["cosine"])

def optimizer_v28(model):
    core, output = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (output if name.startswith("Wo.") else core).append(p)
    params = core + output
    opt = torch.optim.AdamW([
        {"params": core, "lr": 0.003, "base_lr": 0.003},
        {"params": output, "lr": 0.00008, "base_lr": 0.00008},
    ], betas=(0.9, 0.95), weight_decay=2e-4)
    return opt, params

def train_400(model, train_cache, probe_cache, window):
    optimizer, params = optimizer_v28(model)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16, init_scale=128.0)
    best_metrics = local_metrics_from_cache(model, probe_cache)
    best_score = score_local(best_metrics)
    best_step = 0
    best_state = state_cpu(model)
    history = [{"step": 0, **best_metrics}]
    successes, attempts = 0, 0
    print(f"W{window} step    0/{STEPS}: NMSE={best_metrics['nmse']:.5f} cosine={best_metrics['cosine']:.5f}")

    while successes < STEPS:
        if attempts > STEPS + 300:
            raise RuntimeError("Too many AMP overflow/skipped optimizer steps")
        # A moderate cosine decay + short warmup is more stable than PDelta's
        # former 1e-2 opening learning rate, especially with trainable temperature.
        warm = min(1.0, (successes + 1) / 15.0)
        decay = 0.15 + 0.85 * 0.5 * (1.0 + math.cos(math.pi * successes / STEPS))
        for group in optimizer.param_groups:
            group["lr"] = group["base_lr"] * warm * decay

        model.train()
        model.out_drop.eval()
        entry = train_cache[attempts % len(train_cache)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            output = model(entry["x"], **entry["call_kwargs"])[0]
            m = entry["valid"][:, :, None].float()
            target = entry["y"].float()
            pred = output.float()
            mse = (((pred - target) ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            power = ((target ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cos = F.cosine_similarity(pred[entry["valid"]], target[entry["valid"]], dim=-1).mean()
            loss = nmse + 1.50 * (1.0 - cos)
        require_finite(loss, f"V28 W{window} training loss")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        attempts += 1
        if scaler.is_enabled() and scaler.get_scale() < old_scale:
            continue
        successes += 1
        if successes == 1 or successes % CHECK_EVERY == 0 or successes == STEPS:
            probe = local_metrics_from_cache(model, probe_cache)
            history.append({"step": successes, **probe})
            s = score_local(probe)
            if s < best_score:
                best_score, best_metrics, best_step = s, probe, successes
                best_state = state_cpu(model)
            print(
                f"W{window} step {successes:4d}/{STEPS}: "
                f"NMSE={probe['nmse']:.5f} cosine={probe['cosine']:.5f} "
                f"| best cosine={best_metrics['cosine']:.5f} @ {best_step}"
            )
    model.load_state_dict(best_state, strict=True)
    model.eval()
    return {
        "best_step": best_step, "best_probe": best_metrics,
        "attempts": attempts, "successful_updates": successes,
        "history": history,
    }


In [ ]:
#@title 6. Exact CUDA layer timing helpers and original fidelity metrics
@torch.no_grad()
def validate_pair(student, items=dev_items, limit=None):
    teacher.eval()
    student.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    total = agree = teacher_gold = student_gold = 0
    js = []
    for off in range(0, limit, BATCH):
        chosen = items[off:min(off + BATCH, limit)]
        b = direct_batch(chosen)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = student(**b)
        require_finite(tl, "teacher validation logits")
        require_finite(sl, "V28 validation logits")
        allowed = b["marker_mask"].bool()
        tp = torch.softmax(tl.float().masked_fill(~allowed, -1e9), -1)
        sp = torch.softmax(sl.float().masked_fill(~allowed, -1e9), -1)
        gt = torch.tensor([row["gold_index"] for row in chosen], device=device)
        agree += int((tp.argmax(-1) == sp.argmax(-1)).sum())
        teacher_gold += int((tp.argmax(-1) == gt).sum())
        student_gold += int((sp.argmax(-1) == gt).sum())
        mid = 0.5 * (tp + sp)
        q = 0.5 * (tp * (tp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        q += 0.5 * (sp * (sp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        js.extend(q.cpu().tolist())
        total += len(chosen)
    return dict(
        decisions=total, agreement=agree / max(total, 1),
        mean_js=float(np.mean(js)),
        teacher_gold_accuracy=teacher_gold / max(total, 1),
        student_gold_accuracy=student_gold / max(total, 1),
    )

@torch.inference_mode()
def timed_attention(module, entry, warmup=5, repeats=15):
    module.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        for _ in range(warmup):
            module(entry["x"], **entry["call_kwargs"])
        torch.cuda.synchronize()
        ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            module(entry["x"], **entry["call_kwargs"])
            torch.cuda.synchronize()
            ms.append((time.perf_counter() - start) * 1000)
    return float(np.median(ms))

def attention_coverage(cache, window):
    lengths = torch.cat([item["valid"].sum(-1).cpu() for item in cache]).numpy()
    # A centered W=96 attends only 47 tokens left and 48 right.
    # Length <= W alone does NOT guarantee all-to-all attention.
    full_context_limit = min((window - 1) // 2, window // 2) + 1
    return {
        "sequences": int(len(lengths)),
        "median_valid_tokens": float(np.median(lengths)),
        "max_valid_tokens": int(np.max(lengths)),
        "fraction_fully_covered_by_local_window": float(np.mean(lengths <= full_context_limit)),
        "full_context_length_limit": int(full_context_limit),
        "sequences_longer_than_window": int(np.sum(lengths > window)),
        "sequences_at_least_2x_window": int(np.sum(lengths >= 2 * window)),
    }



In [ ]:
#@title 7. Train or restore ONLY global layer 15; all other modules native
from tinycenn_lm.jev_cenn_ablation import (
    JevGlobalBranchAblation, ABLATION_MODES,
    BRANCHES, active_branches, mean_routing_weights
)
assert len(teacher.encoder.layers)==len(student_eval.encoder.layers)
assert LAYER in GLOBAL_LAYERS
original_layer15=student_eval.encoder.layers[LAYER].attn
assert all(teacher.encoder.layers[i].attn is not None for i in range(len(GLOBAL_LAYERS)+len(SLIDING_LAYERS)))
# Caches are captured from ORIGINAL teacher, before any student replacement.
train_cache=cache_teacher_io(LAYER,train_items,TRAIN_CACHE_BATCHES,offset=0)
probe_cache=cache_teacher_io(LAYER,dev_items,PROBE_CACHE_BATCHES,offset=0)
audit_cache=cache_teacher_io(
    LAYER,dev_items,AUDIT_CACHE_BATCHES,offset=PROBE_CACHE_BATCHES*BATCH)
QUALITY_ITEMS=dev_items[(PROBE_CACHE_BATCHES+AUDIT_CACHE_BATCHES)*BATCH:]
assert len(QUALITY_ITEMS)>=QUALITY_DECISIONS, (
    "Increase DEV_CASES or reduce QUALITY_DECISIONS; all splits must be disjoint"
)
trained=build_v28(FULL_WINDOW)
candidate_file=Path(LAYER15_CHECKPOINT)
loaded=False
if LOAD_LAYER15_CHECKPOINT and candidate_file.is_file():
    checkpoint=torch.load(candidate_file,map_location="cpu",weights_only=True)
    assert int(checkpoint["layer"])==LAYER
    assert int(checkpoint["window"])==FULL_WINDOW
    trained.load_state_dict(checkpoint["state_dict"],strict=True)
    loaded=True
    train_summary=checkpoint.get("train",{"source":"existing checkpoint"})
    print("RESTORED trained Jev_CeNN L15 checkpoint:",candidate_file)
else:
    STEPS=GLOBAL_STEPS
    train_summary=train_400(trained,train_cache,probe_cache,FULL_WINDOW)
    path=OUTPUT_DIR/"jev_cenn_l15_trained.pt"
    torch.save({"layer":LAYER,"window":FULL_WINDOW,
                "architecture":"pdelta_v28_fast_window_memory",
                "train":train_summary,"state_dict":state_cpu(trained)},path)
    print("TRAINED and saved ONLY L15:",path)
trained.eval().requires_grad_(False)
base_probe=local_metrics_from_cache(trained,probe_cache)
base_audit=local_metrics_from_cache(trained,audit_cache)
print("L15 trained base probe:",base_probe)
print("L15 trained base AUDIT:",base_audit)
assert all(
    student_eval.encoder.layers[i].attn is not trained
    for i in range(len(student_eval.encoder.layers)) if i!=LAYER
)
assert all(type(student_eval.encoder.layers[i].attn)
           is type(teacher.encoder.layers[i].attn)
           for i in range(len(student_eval.encoder.layers)) if i!=LAYER)
print("All 27 other encoder attention layers unchanged: PASS")


In [ ]:
#@title 8. Verify full/manual forward parity; inspect learned branch-routing weights
sample=audit_cache[0]
full=JevGlobalBranchAblation(trained,"full").to(device).eval()
manual=JevGlobalBranchAblation(trained,"manual_full").to(device).eval()
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    exact=full(sample["x"],**sample["call_kwargs"])[0]
    replay=manual(sample["x"],**sample["call_kwargs"])[0]
valid=sample["valid"].bool()
delta=float((exact.float()[valid]-replay.float()[valid]).abs().max())
cos=float(F.cosine_similarity(exact.float()[valid],
                              replay.float()[valid],dim=-1).mean())
if not(delta<=0.004 and cos>=0.99999):
    raise RuntimeError(f"ABORT: ablation forward differs from trained V3.1! "
                       f"maxΔ={delta:.6g}, cosine={cos:.7f}")
print(f"FULL vs manually reconstructed L15: PASS maxΔ={delta:.7f} cosine={cos:.7f}")
route_usage=mean_routing_weights(trained,audit_cache)
print("Mean trained L15 route probabilities (across valid tokens & heads):")
for name,p in route_usage.items():
    print(f"  {name:<10s}  {p:.4%}")
assert abs(sum(route_usage.values())-1.0)<0.005


In [ ]:
#@title 9. Cache original-teacher DEV decisions ONCE; then measure all surgical ablations
# Teacher logits cached on the DOWNSTREAM quality split only. The local
# input/output audit partition is different and never used for training.
quality_rows=QUALITY_ITEMS[:QUALITY_DECISIONS]
eval_batches=[]
for start in range(0,len(quality_rows),BATCH):
    chosen=quality_rows[start:start+BATCH]
    b=direct_batch(chosen)
    with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**b)
    eval_batches.append({
        "batch":b,
        "reference_logits":teacher_logits.float().detach(),
        "gold":torch.tensor([r["gold_index"] for r in chosen],device=device),
    })
print("Independent downstream DEV items:",len(quality_rows))
@torch.inference_mode()
def decisions_and_accuracy():
    total=agree=correct_teacher=correct_student=0
    js_total=0.0
    for e in eval_batches:
        inp=e["batch"]
        with torch.autocast("cuda",dtype=amp_dtype):
            student_logits,_=student_eval(**inp)
        allowed=inp["marker_mask"].bool()
        tp=torch.softmax(e["reference_logits"].masked_fill(~allowed,-1e9),-1)
        sp=torch.softmax(student_logits.float().masked_fill(~allowed,-1e9),-1)
        ti=tp.argmax(-1)
        si=sp.argmax(-1)
        agree+=int((ti==si).sum())
        correct_teacher+=int((ti==e["gold"]).sum())
        correct_student+=int((si==e["gold"]).sum())
        mid=0.5*(tp+sp)
        divergence=0.5*(tp*(tp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        divergence+=0.5*(sp*(sp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        js_total+=float(divergence.sum())
        total+=len(ti)
    return {
        "decisions":total,"agreement":agree/total,
        "teacher_gold_accuracy":correct_teacher/total,
        "student_gold_accuracy":correct_student/total,
        "mean_js":js_total/total,
    }
@torch.inference_mode()
def timed_entire_model(m,inp,repeats=LATENCY_REPEATS):
    with torch.autocast("cuda",dtype=amp_dtype):
        for _ in range(5):m(**inp)
        torch.cuda.synchronize()
        observed=[]
        for _ in range(repeats):
            start=time.perf_counter()
            m(**inp)
            torch.cuda.synchronize()
            observed.append((time.perf_counter()-start)*1000)
    return {"median_ms":float(np.median(observed)),
            "p95_ms":float(np.percentile(observed,95))}
native_layer_ms=timed_attention(teacher.encoder.layers[LAYER].attn,
                                audit_cache[0],repeats=LATENCY_REPEATS)
student_eval.eval().requires_grad_(False)
native_full_ms=timed_entire_model(teacher,eval_batches[0]["batch"])["median_ms"]
RESULTS=[]
for mode in ABLATION_MODES:
    module=JevGlobalBranchAblation(trained,mode).to(device).eval()
    student_eval.encoder.layers[LAYER].attn=module
    local=local_metrics_from_cache(module,audit_cache)
    quality=decisions_and_accuracy()
    layer_ms=timed_attention(module,audit_cache[0],repeats=LATENCY_REPEATS)
    full_time=timed_entire_model(student_eval,eval_batches[0]["batch"])
    item={
        "mode":mode,"layer":LAYER,"active":list(active_branches(mode)),
        "local":local,"task":quality,"layer_median_ms":layer_ms,
        "whole_model_median_ms":full_time["median_ms"],
        "whole_model_p95_ms":full_time["p95_ms"],
        "native_layer_ms":native_layer_ms,
        "native_model_ms":native_full_ms,
        "layer_speedup_over_native":native_layer_ms/max(layer_ms,1e-8),
        "whole_speedup_over_native":native_full_ms/max(full_time["median_ms"],1e-8),
        "train_free":True,
    }
    RESULTS.append(item)
    print(f"{mode:>17s}: cosine={local['cosine']:.6f} "
          f"NMSE={local['nmse']:.5f} "
          f"agreement={quality['agreement']:.4f} "
          f"gold={quality['student_gold_accuracy']:.4f} "
          f"layer={layer_ms:.3f}ms e2e={full_time['median_ms']:.3f}ms "
          f"e2e/native={item['whole_speedup_over_native']:.3f}x")
    del module
student_eval.encoder.layers[LAYER].attn=original_layer15
baseline=next(r for r in RESULTS if r["mode"]=="full")
for result in RESULTS:
    result["delta_cosine_vs_full"]=result["local"]["cosine"]-baseline["local"]["cosine"]
    result["delta_gold_pp_vs_full"]=100.0*(
        result["task"]["student_gold_accuracy"]-baseline["task"]["student_gold_accuracy"]
    )
    result["delta_agreement_pp_vs_full"]=100.0*(
        result["task"]["agreement"]-baseline["task"]["agreement"]
    )
    result["e2e_gain_over_full"]=baseline["whole_model_median_ms"]/max(
        result["whole_model_median_ms"],1e-8
    )
assert len(RESULTS)==len(ABLATION_MODES)
assert abs(next(x for x in RESULTS if x["mode"]=="manual_full")["local"]["cosine"]-
           baseline["local"]["cosine"])<1e-5
print("One-layer zero-retraining ablation study complete:",len(RESULTS),"conditions")


In [ ]:
#@title 10. OPTIONAL recovery learning: can a removed branch be compensated?
# Recovers each mode from the SAME trained full checkpoint, never sequentially
# from a previously ablated model. Probe selects checkpoints, audit does not.
RECOVERY=[]
if RECOVERY_STEPS>0:
    for mode in RECOVERY_MODES:
        print(f"\nRECOVERY {mode} / {RECOVERY_STEPS} optimizer steps")
        recovered=JevGlobalBranchAblation(copy.deepcopy(trained),mode).to(device)
        for name,p in recovered.named_parameters():
            p.requires_grad_(not name.startswith(
                ("base.Wqkv.","base.rotary_emb.")))
        recovered.train()
        recovered.base.out_drop.eval()
        candidate=[p for p in recovered.parameters() if p.requires_grad]
        wo=[p for name,p in recovered.named_parameters()
            if name.startswith("base.Wo.") and p.requires_grad]
        other=[p for name,p in recovered.named_parameters()
               if not name.startswith("base.Wo.") and p.requires_grad]
        optim=torch.optim.AdamW([
            {"params":other,"lr":0.0005},
            {"params":wo,"lr":0.00002},
        ],weight_decay=2e-4)
        scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
        start_probe=local_metrics_from_cache(recovered,probe_cache)
        best_metric=start_probe
        best_score=score_local(start_probe)
        best_state=state_cpu(recovered)
        best_step=0
        successful=0
        attempts=0
        while successful<RECOVERY_STEPS:
            if attempts>RECOVERY_STEPS+250:
                raise RuntimeError(f"Recovery repeated AMP overflow: {mode}")
            entry=train_cache[attempts%len(train_cache)]
            optim.zero_grad(set_to_none=True)
            recovered.train()
            recovered.base.out_drop.eval()
            with torch.autocast("cuda",dtype=amp_dtype):
                predicted=recovered(entry["x"],**entry["call_kwargs"])[0]
                valid=entry["valid"]
                m=valid[:,:,None].float()
                target=entry["y"].float()
                diff=((predicted.float()-target)**2*m).sum()
                normalizer=(target.square()*m).sum().clamp_min(1e-8)
                nmse=diff/normalizer
                cosine=F.cosine_similarity(predicted.float()[valid],
                                           target[valid],dim=-1).mean()
                loss=nmse+1.5*(1-cosine)
            require_finite(loss,f"Jev_CeNN recovery {mode}")
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(candidate,1.0)
            scale_before=scaler.get_scale()
            scaler.step(optim)
            scaler.update()
            attempts+=1
            if scaler.is_enabled() and scaler.get_scale()<scale_before:
                continue
            successful+=1
            if successful==1 or successful%50==0 or successful==RECOVERY_STEPS:
                metrics=local_metrics_from_cache(recovered,probe_cache)
                if score_local(metrics)<best_score:
                    best_metric=metrics
                    best_score=score_local(metrics)
                    best_step=successful
                    best_state=state_cpu(recovered)
                print(f"  {successful:4d}: probe cos={metrics['cosine']:.6f}, "
                      f"NMSE={metrics['nmse']:.5f}, best@{best_step}")
        recovered.load_state_dict(best_state,strict=True)
        recovered.eval().requires_grad_(False)
        # Only NOW consult blind audit and independent downstream quality.
        blind=local_metrics_from_cache(recovered,audit_cache)
        student_eval.encoder.layers[LAYER].attn=recovered
        task=decisions_and_accuracy()
        ms=timed_attention(recovered,audit_cache[0],repeats=LATENCY_REPEATS)
        e2e=timed_entire_model(student_eval,eval_batches[0]["batch"])
        out=OUTPUT_DIR/f"l15_recovery_{mode}.pt"
        torch.save({"mode":mode,"layer":LAYER,"steps":RECOVERY_STEPS,
                    "best_step":best_step,"probe":best_metric,
                    "audit":blind,"state_dict":state_cpu(recovered)},out)
        RECOVERY.append({
            "mode":mode,"best_step":best_step,"probe":best_metric,
            "audit":blind,"task":task,"layer_median_ms":ms,
            "whole_model_median_ms":e2e["median_ms"],
            "checkpoint":str(out)})
        print(f"  BLIND AUDIT cos={blind['cosine']:.6f}, "
              f"agreement={task['agreement']:.4f}, "
              f"gold={task['student_gold_accuracy']:.4f}")
        student_eval.encoder.layers[LAYER].attn=original_layer15
        del recovered
        torch.cuda.empty_cache()
else:
    print("RECOVERY_STEPS=0: clean frozen-weight ablations only.")


In [ ]:
#@title 11. Report contribution magnitudes, quality/speed Pareto plot and CSV/JSON
import csv
import matplotlib.pyplot as plt
summary_rows=[]
for row in RESULTS:
    summary_rows.append({
        "mode":row["mode"],
        "audit_cosine":row["local"]["cosine"],
        "audit_nmse":row["local"]["nmse"],
        "teacher_agreement":row["task"]["agreement"],
        "gold_accuracy":row["task"]["student_gold_accuracy"],
        "delta_agreement_pp_vs_full":row["delta_agreement_pp_vs_full"],
        "delta_gold_pp_vs_full":row["delta_gold_pp_vs_full"],
        "layer_ms":row["layer_median_ms"],
        "whole_ms":row["whole_model_median_ms"],
        "e2e_speedup_vs_full":row["e2e_gain_over_full"],
        "e2e_speedup_vs_teacher":row["whole_speedup_over_native"]
    })
csv_file=OUTPUT_DIR/"jev_cenn_l15_surgical_ablation.csv"
with csv_file.open("w",newline="") as handle:
    writer=csv.DictWriter(handle,fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)
report={
    "model":"Jev_CeNN","source_teacher":SOURCE_MODEL,
    "scope":"one original full-attention encoder layer 15 only",
    "encoder_layers":len(teacher.encoder.layers),
    "modified_layers":[LAYER],
    "all_other_attention_layers_native":True,
    "official_test_split_used":False,
    "trained_checkpoint_reused":loaded,
    "trained_full_probe":base_probe,"trained_full_blind_audit":base_audit,
    "manual_full_max_abs_error":delta,
    "manual_full_cosine":cos,"branch_route_usage":route_usage,
    "native_layer_ms":native_layer_ms,
    "native_model_ms":native_full_ms,
    "quality_decisions":len(quality_rows),
    "intervention":"drop-one with routing mass renormalization",
    "methodology_note":"marginal branch effects interact; they are not Shapley values",
    "train_free":RESULTS,"recovery":RECOVERY,
}
json_file=OUTPUT_DIR/"jev_cenn_l15_ablation_report.json"
json_file.write_text(json.dumps(report,indent=2,allow_nan=False))
fig,axes=plt.subplots(1,2,figsize=(15,6))
labels=[r["mode"] for r in summary_rows]
axes[0].barh(labels,[r["audit_cosine"] for r in summary_rows])
axes[0].axvline(0.99,ls="--",color="gray",label="Target 0.99")
axes[0].set(xlabel="Blind-audit L15 cosine",title="Local output fidelity")
axes[0].legend()
axes[1].scatter([r["whole_ms"] for r in summary_rows],
                [r["teacher_agreement"] for r in summary_rows])
for row in summary_rows:
    axes[1].annotate(row["mode"],(row["whole_ms"],row["teacher_agreement"]),
                     fontsize=8,xytext=(4,3),textcoords="offset points")
axes[1].axhline(0.99,ls="--",color="gray")
axes[1].set(xlabel="Full-model median CUDA latency (ms)",
            ylabel="Teacher agreement",title="Quality / speed tradeoff")
axes[1].grid(alpha=0.2)
fig.tight_layout()
figure=OUTPUT_DIR/"jev_cenn_l15_ablation.png"
fig.savefig(figure,dpi=170,bbox_inches="tight")
plt.show()
print("\nFASTEST SURGICAL VARIANTS, VALIDATION:")
for row in sorted(summary_rows,key=lambda r:r["whole_ms"]):
    print(f" {row['mode']:<17} {row['whole_ms']:8.2f} ms | "
          f"cos={row['audit_cosine']:.6f} | "
          f"agreement={row['teacher_agreement']:.4f} | "
          f"Δgold={row['delta_gold_pp_vs_full']:+.2f} pp")
print("Saved CSV:",csv_file)
print("Saved JSON:",json_file)
print("Saved figure:",figure)
print("No Hugging Face upload. All observations are one-L15 exploratory DEV results.")
